In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- load data ---
DATA_DIR = Path("../data/raw")
train = pd.read_csv(DATA_DIR / "train.csv")

target = "SalePrice"
id_col = "Id"

y = train[target]
X = train.drop(columns=[target, id_col])

# --- local split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- feature groups ---
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

# --- preprocessing ---
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# --- final candidate ---
hgb = HistGradientBoostingRegressor(
    l2_regularization=0.1,
    learning_rate=0.1,
    max_leaf_nodes=15,
    random_state=42
)

final_model = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", preprocessor),
        ("model", hgb)
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

# --- fit on X_train ---
final_model.fit(X_train, y_train)

# --- evaluate on X_test ---
y_pred = final_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
rmsle = np.sqrt(mean_squared_error(np.log1p(y_test), np.log1p(np.maximum(y_pred,0))))

print(f"Final test metrics:")
print(f"MAE = {mae:.2f}")
print(f"RMSE = {rmse:.2f}")
print(f"R² = {r2:.4f}")
print(f"RMSLE = {rmsle:.4f}")

C:\Users\Александр\AppData\Local\Temp\ipykernel_24456\2465993873.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()


Final test metrics:
MAE = 15830.01
RMSE = 27974.15
R² = 0.8980
RMSLE = 0.1332


- Test metrics vs CV expectations
- Signs of overfitting/instability
- Final honest model quality
- Limitations of RMSE-priority choice
- Note: GradientBoostingRegressor remains an alternative by MAE/RMSLE but was not evaluated

1. Train vs test error

In [2]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        "RMSLE": np.sqrt(
            mean_squared_error(
                np.log1p(y_true),
                np.log1p(np.maximum(y_pred, 0))
            )
        )
    }

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

train_metrics = regression_metrics(y_train, y_train_pred)
test_metrics = regression_metrics(y_test, y_test_pred)

train_test_comparison = pd.DataFrame([
    {"split": "train", **train_metrics},
    {"split": "test", **test_metrics},
])

display(train_test_comparison)

,split,MAE,RMSE,R2,RMSLE
0,train,7506.365706,12380.061035,0.974304,0.061856
1,test,15830.009639,27974.145853,0.897977,0.133197


In [3]:
baseline_metrics = {
    "MAE": 18758.16,
    "RMSE": 33857.89,
    "R2": 0.8029,
}

In [4]:
final_test_metrics = {
    "MAE": test_metrics["MAE"],
    "RMSE": test_metrics["RMSE"],
    "R2": test_metrics["R2"],
}

improvement = pd.DataFrame([
    {
        "metric": "MAE",
        "baseline_cv": baseline_metrics["MAE"],
        "final_test": final_test_metrics["MAE"],
        "absolute_improvement": baseline_metrics["MAE"] - final_test_metrics["MAE"],
        "relative_improvement_pct": (
            (baseline_metrics["MAE"] - final_test_metrics["MAE"])
            / baseline_metrics["MAE"]
            * 100
        ),
    },
    {
        "metric": "RMSE",
        "baseline_cv": baseline_metrics["RMSE"],
        "final_test": final_test_metrics["RMSE"],
        "absolute_improvement": baseline_metrics["RMSE"] - final_test_metrics["RMSE"],
        "relative_improvement_pct": (
            (baseline_metrics["RMSE"] - final_test_metrics["RMSE"])
            / baseline_metrics["RMSE"]
            * 100
        ),
    },
    {
        "metric": "R2",
        "baseline_cv": baseline_metrics["R2"],
        "final_test": final_test_metrics["R2"],
        "absolute_improvement": final_test_metrics["R2"] - baseline_metrics["R2"],
        "relative_improvement_pct": (
            (final_test_metrics["R2"] - baseline_metrics["R2"])
            / baseline_metrics["R2"]
            * 100
        ),
    },
])

display(improvement)

,metric,baseline_cv,final_test,absolute_improvement,relative_improvement_pct
0,MAE,18758.1600,15830.009639,2928.150361,15.610008
1,RMSE,33857.8900,27974.145853,5883.744147,17.377764
2,R2,0.8029,0.897977,0.095077,11.841648


This comparison is approximate because Ridge baseline metrics are CV metrics on X_train, while final metrics are from the held-out X_test. It is useful as a high-level progress summary, not as a strict apples-to-apples statistical comparison.

Train error is much lower than test error, which indicates that the final boosting model fits the training data substantially better than unseen data. This is expected for flexible tree boosting models and shows some overfitting capacity.

However, final test metrics are very close to Stage 5 cross-validation metrics:
- CV RMSE ≈ 28,110
- Test RMSE ≈ 27,974

Therefore, there is no evidence that model selection overfit the validation process. The held-out test set confirms the CV estimate. The model generalizes reasonably, but the train/test gap suggests that stronger regularization, simpler models, or additional tuning could be explored in future work, not after final evaluation.

Ridge baseline was CV on X_train.
Final result is one-time holdout test.
Comparison is directional, not perfectly apples-to-apples.

## Save final model artifact

The final model is fitted on X_train/y_train only using the frozen policy approved after final evaluation. The local X_test/y_test is not re-evaluated here.

In [5]:
import joblib
from pathlib import Path

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

final_artifact_path = MODELS_DIR / "final_house_price_pipeline.joblib"

# Refit frozen final model on X_train / y_train only.
# Do not evaluate again on X_test.
final_model.fit(X_train, y_train)

joblib.dump(final_model, final_artifact_path)

print("Saved to:", final_artifact_path)
print("Exists:", final_artifact_path.exists())
print("Size bytes:", final_artifact_path.stat().st_size)

Saved to: ..\models\final_house_price_pipeline.joblib
Exists: True
Size bytes: 271361


In [6]:
loaded_model = joblib.load(final_artifact_path)

sample_X = X_train.head(5)
sample_predictions = loaded_model.predict(sample_X)

print("Sample predictions:")
print(sample_predictions)

print("Prediction shape:", sample_predictions.shape)
print("Any negative predictions:", (sample_predictions < 0).any())

Sample predictions:
[146199.57926824 174233.06196705  93506.03502322 161468.65815998
 139510.78155702]
Prediction shape: (5,)
Any negative predictions: False
